# WakeMarch: resolution and computational cost

Run all cells from a fresh kernel. This data-free example compares D/10, D/20 and D/30 on the same physical sampling mesh. D/30 is a finer numerical reference, **not ground truth or experimental validation**. Model constants stay fixed.

These are interactive, single-run timings including progress display. For speed comparisons use fresh processes, progress off, repeated runs, and profile separately. Do not compare these times directly with timings from another machine or session.

The timings use the direct scalar-CT `WakeModel` with fixed inputs. Turbine curves and fixed-CP power estimates are not part of this resolution comparison; see `wakemarch_turbines.ipynb` for those input modes.

In [ ]:
import json
import platform
import time
import numpy as np
import scipy
import matplotlib.pyplot as plt
import wakemarch
from wakemarch import WakeModel, Numerics

print("WakeMarch", wakemarch.__version__, wakemarch.__file__)
print(platform.platform(), "NumPy", np.__version__, "SciPy", scipy.__version__)
params = dict(D=126., H=90., U=8., CT=0.8, yaw_deg=25., TI=0.06)
resolutions = (10, 20, 30)
y, z = np.linspace(-252, 252, 101), np.linspace(0, 279, 81)
yy, zz = np.meshgrid(y, z)
planes, records = {}, []

## Run the same wake on three grids

Each bar includes the downstream buffer needed for yawed reconstruction. The stored-plane request is 0.25D; with marching steps of 0.1D the actual interval is 0.3D. The model is reconstructed at 3D, 6D and 10D below. Timings include construction, marching and sampling; imports are excluded.

In [ ]:
for cells_per_D in resolutions:
    print(f"Grid D/{cells_per_D}")
    start = time.perf_counter()
    model = WakeModel(**params, numerics=Numerics(h_D=1/cells_per_D))
    result = model.solve(10, progress=True)
    sample = np.stack([result.sample_deficit(x*model.D, yy, zz) for x in (3, 6, 10)])
    elapsed = time.perf_counter()-start
    assert sample.shape == (3, 81, 101) and np.isfinite(sample).all()
    assert np.allclose(np.diff(result.x)/model.D, 0.3)
    planes[cells_per_D] = sample
    records.append(dict(cells_per_D=cells_per_D, seconds=elapsed,
        history_MiB=sum(a.nbytes for a in (result.marching_deficit,
            result.streamwise_vorticity, result.streamwise_velocity_derivative))/1024**2))
    del result

## Compare numerical differences and cost

The discrepancy is RMS of deficit differences over all three planes, divided by hub-height inflow speed. Peak difference is also shown. These metrics are grid sensitivity for this case; they are not observational error. Retained history memory excludes solver caches, temporary allocations and interpreter memory; the command-line benchmark reports process peak RSS separately.

In [ ]:
reference = planes[30]
for row in records:
    delta = planes[row['cells_per_D']] - reference
    row['rms_difference_over_U'] = float(np.sqrt(np.mean(delta**2))/params['U'])
    row['peak_difference_m_per_s'] = float(np.max(np.abs(delta)))
assert records[-1]['rms_difference_over_U'] == 0
print(json.dumps(dict(version=wakemarch.__version__, parameters=params, results=records), indent=2))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(resolutions, [r['seconds'] for r in records], 'o-')
axes[0].set(xlabel='Grid cells per diameter', ylabel='Interactive runtime [s]')
axes[1].plot(resolutions, [r['rms_difference_over_U'] for r in records], 'o-')
axes[1].set(xlabel='Grid cells per diameter', ylabel='RMS difference from D/30 / U')
for ax in axes:
    ax.set_xticks(resolutions)
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

Try changing yaw, turbulence intensity or shear and rerun all cells. High yaw with shear is the most costly diffusion case among the reference cases. Refining the grid increases cost; a zero discrepancy at D/30 simply means it is the reference. A convergence study needs additional refinement and checks of domain size, marching step and the physical outputs of interest.

The package uses cached geometry and diffusion factorizations. No JIT compiler is required and there is no JIT compilation warm-up. First-import costs and machine load still affect interactive experience. Equivalence with the research solver is checked separately (see `PROVENANCE.md`).